# Source Data Model in Full Refresh



Dit notebook voert de Full Refresh strategie uit:

- Stap 1: Alle SDM-tabellen leegmaken (DELETE)

- Stap 2: Data uit alle 5 operationele DB's opnieuw inlezen in het SDM met INSERT



Inlaadstrategie voor SCD:

- SDM: full refresh. Het SDM bewaart geen historie en voert zelf geen SCD Type 1 of Type 2 uit.

- DWH SCD Type 1: gewijzigde dimensierij met dezelfde Business Key wordt overschreven.

- DWH SCD Type 2: oude versie krijgt een eindtijd en een nieuwe versie krijgt een nieuwe Surrogate Key.

- Feiten bij SCD Type 2: een nieuwe feitrij gebruikt in het DWH de meest actuele Surrogate Key die hoort bij de Business Key uit het SDM.



Opdrachtcontrole:

- Ja, deze opdracht is al afgedekt in DWH.ipynb.

- Test 3 wijzigt eerst de klant met Business Key `klantnr = 1`.

- Test 5 voegt daarna een nieuwe feitrij toe in het SDM die weer verwijst naar `klantnr = 1`.

- Bij een SCD Type 2 DWH krijgt die nieuwe feitrij daarna de nieuwste actuele `klant_sk`.

- Die surrogate-key verwachting geldt alleen wanneer het actieve DWH op het SCD Type 2 schema draait.



**Let op**: draai dit notebook van boven naar beneden voor een volledige bron-refresh. Voor de SCD-validatie gebruik je daarna DWH.ipynb Test 3 en Test 5.


In [1]:
# 1. IMPORTS EN INSTELLINGEN
import sqlite3
import logging
import pandas as pd
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# Database paden
DB_ACCESSOIREVERKOOP = "BikeToDrive_1_Accessoireverkoop.db"
DB_FIETSVERKOOP      = "BikeToDrive_2_Fietsverkoop.db"
DB_ONDERHOUD         = "BikeToDrive_3_Onderhoud.db"
DB_ACCESSOIRE_INKOOP = "BikeToDrive_4_Accessoire_Inkoop.db"
DB_FIETS_INKOOP      = "BikeToDrive_5_Fiets_Inkoop.db"
DB_SDM               = "BikeToDrive_6 _SourceDataModel.db"

log.info(f"Imports gedaan — SDM: {DB_SDM}")


INFO | Imports gedaan — SDM: BikeToDrive_6 _SourceDataModel.db


In [2]:
# 2. TABLE MAPPING CONFIGURATIE
# Elke tuple: (bron_db, bron_tabel, sdm_tabel)
# Volgorde = insert-volgorde (ouder-tabellen eerst voor FK-veiligheid)

TABLE_MAPPING = [
    # Accessoireverkoop
    (DB_ACCESSOIREVERKOOP, "Leverancier",       "Accessoire_Verkoop_Leverancier"),
    (DB_ACCESSOIREVERKOOP, "Accessoire",         "Accessoire_Verkoop_Accessoire"),
    (DB_ACCESSOIREVERKOOP, "Filiaal",            "Accessoire_Verkoop_Filiaal"),
    (DB_ACCESSOIREVERKOOP, "Monteur",            "Accessoire_Verkoop_Monteur"),
    (DB_ACCESSOIREVERKOOP, "Klant",              "Accessoire_Verkoop_Klant"),
    (DB_ACCESSOIREVERKOOP, "Accessoire_Verkoop", "Accessoire_Verkoop"),

    # Fietsverkoop
    (DB_FIETSVERKOOP, "Fabrikant",    "Fiets_Verkoop_Fabrikant"),
    (DB_FIETSVERKOOP, "Fiets",        "Fiets_Verkoop_Fiets"),
    (DB_FIETSVERKOOP, "Filiaal",      "Fiets_Verkoop_Filiaal"),
    (DB_FIETSVERKOOP, "Monteur",      "Fiets_Verkoop_Monteur"),
    (DB_FIETSVERKOOP, "Klant",        "Fiets_Verkoop_Klant"),
    (DB_FIETSVERKOOP, "Fiets_Verkoop","Fiets_Verkoop"),

    # Onderhoud
    (DB_ONDERHOUD, "Fabrikant", "Onderhoud_Fabrikant"),
    (DB_ONDERHOUD, "Fiets",     "Onderhoud_Fiets"),
    (DB_ONDERHOUD, "Filiaal",   "Onderhoud_Filiaal"),
    (DB_ONDERHOUD, "Monteur",   "Onderhoud_Monteur"),
    (DB_ONDERHOUD, "Onderhoud", "Onderhoud"),

    # Accessoire Inkoop
    (DB_ACCESSOIRE_INKOOP, "Leverancier",      "Accessoire_Inkoop_Leverancier"),
    (DB_ACCESSOIRE_INKOOP, "Accessoire",        "Accessoire_Inkoop_Accessoire"),
    (DB_ACCESSOIRE_INKOOP, "Accessoire_Inkoop", "Accessoire_Inkoop"),

    # Fiets Inkoop
    (DB_FIETS_INKOOP, "Fabrikant",    "Fiets_Inkoop_Fabrikant"),
    (DB_FIETS_INKOOP, "Fiets",        "Fiets_Inkoop_Fiets"),
    (DB_FIETS_INKOOP, "Fiets_Inkoop", "Fiets_Inkoop"),
]

log.info(f"TABLE_MAPPING gedefinieerd — {len(TABLE_MAPPING)} tabellen geconfigureerd")


INFO | TABLE_MAPPING gedefinieerd — 23 tabellen geconfigureerd


In [3]:
# 3. GENERIEKE FUNCTIES

def reset_sdm(conn):
    """Verwijdert alle data uit SDM tabellen (omgekeerde insert-volgorde voor FK-veiligheid)."""
    for _, _, sdm_tabel in reversed(TABLE_MAPPING):
        try:
            conn.execute(f"DELETE FROM {sdm_tabel}")
            log.info(f"✓ reset: {sdm_tabel}")
        except sqlite3.Error as e:
            log.error(f"✗ reset {sdm_tabel}: {e}")
    conn.commit()


def load_db_to_sdm(sdm_conn):
    """Laadt alle bron-tabellen in het SDM via TABLE_MAPPING (pandas to_sql)."""
    huidig_db = None
    for src_db, src_tabel, sdm_tabel in TABLE_MAPPING:
        if src_db != huidig_db:
            log.info(f"--- Laden uit: {src_db} ---")
            huidig_db = src_db
        try:
            with sqlite3.connect(src_db) as src_conn:
                df = pd.read_sql_query(f"SELECT * FROM {src_tabel}", src_conn)
            df.to_sql(sdm_tabel, sdm_conn, if_exists="append", index=False)
            log.info(f"✓ {src_tabel} → {sdm_tabel} ({len(df)} rijen)")
        except Exception as e:
            log.error(f"✗ {src_tabel} → {sdm_tabel}: {e}")
    sdm_conn.commit()


log.info("Generieke functies gedefinieerd (reset_sdm, load_db_to_sdm)")


INFO | Generieke functies gedefinieerd (reset_sdm, load_db_to_sdm)


In [4]:
# 4. MAIN SCRIPT - Full Refresh uitvoeren

log.info("=" * 50)
log.info("START FULL REFRESH VAN SOURCE DATA MODEL")
log.info("=" * 50)

try:
    with sqlite3.connect(DB_SDM) as sdm_conn:
        log.info("STAP 1: Reset — alle SDM tabellen leegmaken")
        reset_sdm(sdm_conn)

        log.info("STAP 2: Data laden van operationele databases")
        load_db_to_sdm(sdm_conn)

    log.info("=" * 50)
    log.info("FULL REFRESH SUCCESVOL AFGEROND")
    log.info("=" * 50)

except Exception as e:
    log.exception(f"FOUT OPGETREDEN tijdens full refresh: {e}")


INFO | ==================================================
INFO | START FULL REFRESH VAN SOURCE DATA MODEL
INFO | ==================================================
INFO | STAP 1: Reset — alle SDM tabellen leegmaken
INFO | ✓ reset: Fiets_Inkoop
INFO | ✓ reset: Fiets_Inkoop_Fiets
INFO | ✓ reset: Fiets_Inkoop_Fabrikant
INFO | ✓ reset: Accessoire_Inkoop
INFO | ✓ reset: Accessoire_Inkoop_Accessoire
INFO | ✓ reset: Accessoire_Inkoop_Leverancier
INFO | ✓ reset: Onderhoud
INFO | ✓ reset: Onderhoud_Monteur
INFO | ✓ reset: Onderhoud_Filiaal
INFO | ✓ reset: Onderhoud_Fiets
INFO | ✓ reset: Onderhoud_Fabrikant
INFO | ✓ reset: Fiets_Verkoop
INFO | ✓ reset: Fiets_Verkoop_Klant
INFO | ✓ reset: Fiets_Verkoop_Monteur
INFO | ✓ reset: Fiets_Verkoop_Filiaal
INFO | ✓ reset: Fiets_Verkoop_Fiets
INFO | ✓ reset: Fiets_Verkoop_Fabrikant
INFO | ✓ reset: Accessoire_Verkoop
INFO | ✓ reset: Accessoire_Verkoop_Klant
INFO | ✓ reset: Accessoire_Verkoop_Monteur
INFO | ✓ reset: Accessoire_Verkoop_Filiaal
INFO | ✓ reset:



##  Reset SDM functie uitvoeren


In [ ]:
# Reset SDM (alleen leegmaken, geen data inladen)

log.info("START RESET SDM")

try:
    with sqlite3.connect(DB_SDM) as sdm_conn:
        reset_sdm(sdm_conn)
    log.info("RESET VOLTOOID — SDM IS LEEG")

except Exception as e:
    log.exception(f"FOUT OPGETREDEN tijdens reset: {e}")




##  Check of data correct in SDM zit


In [5]:
# 5. VERIFICATIE - Aantal rijen per tabel als DataTable

log.info("Verificatie gestart...")

with sqlite3.connect(DB_SDM) as sdm_conn:
    rows = []
    for _, _, sdm_tabel in TABLE_MAPPING:
        try:
            count = pd.read_sql_query(
                f"SELECT COUNT(*) AS aantal FROM {sdm_tabel}", sdm_conn
            ).iloc[0, 0]
            rows.append({"Tabel": sdm_tabel, "Rijen": count, "Status": "✓" if count > 0 else "⚠ leeg"})
        except Exception as e:
            rows.append({"Tabel": sdm_tabel, "Rijen": "–", "Status": f"✗ {e}"})

df_check = pd.DataFrame(rows)
display(df_check.style.set_caption("SDM Verificatie — Rijen per tabel").hide(axis="index"))

log.info(f"Verificatie afgerond — {len(rows)} tabellen gecontroleerd")


INFO | Verificatie gestart...


Tabel,Rijen,Status
Accessoire_Verkoop_Leverancier,5,✓
Accessoire_Verkoop_Accessoire,10,✓
Accessoire_Verkoop_Filiaal,4,✓
Accessoire_Verkoop_Monteur,10,✓
Accessoire_Verkoop_Klant,20,✓
Accessoire_Verkoop,100,✓
Fiets_Verkoop_Fabrikant,10,✓
Fiets_Verkoop_Fiets,75,✓
Fiets_Verkoop_Filiaal,4,✓
Fiets_Verkoop_Monteur,10,✓


INFO | Verificatie afgerond — 23 tabellen gecontroleerd


In [ ]:
# 6. PREVIEW - Bekijk de eerste rijen van een SDM tabel
# Wijzig PREVIEW_TABLE naar de tabel die je wilt inspecteren

PREVIEW_TABLE = "Fiets_Inkoop"

with sqlite3.connect(DB_SDM) as sdm_conn:
    df_preview = pd.read_sql_query(f"SELECT * FROM {PREVIEW_TABLE} LIMIT 10", sdm_conn)

log.info(f"Preview '{PREVIEW_TABLE}' — {len(df_preview)} rijen getoond")
display(df_preview.style.set_caption(f"Preview: {PREVIEW_TABLE} (eerste 10 rijen)").hide(axis="index"))
